In [2]:
import os
import glob
import random
import imageio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [24]:
# Pega a pasta base do projeto
BASE_DIR = os.path.dirname(os.path.abspath(""))
dataset_path = os.path.join(BASE_DIR, "dev_caption")
images_path = os.path.join(dataset_path, "images")

captions_csv = pd.read_csv(f"{dataset_path}/captions.csv")

train_captions = captions_csv.loc[[i for i,x in enumerate(captions_csv['ID']) if "train" in x]]
valid_captions = captions_csv.loc[[i for i,x in enumerate(captions_csv['ID']) if "valid" in x]]

train_captions.to_csv(os.path.join(dataset_path, "train_captions.csv"), index = False)
valid_captions.to_csv(os.path.join(dataset_path, "valid_captions.csv"), index = False)

In [30]:
import shutil

destino_train = os.path.join(dataset_path, "train")
destino_valid = os.path.join(dataset_path, "valid")

count = 0
total = len(os.listdir(images_path))

for arquivo in os.listdir(images_path):
    if "train" in arquivo:
        shutil.move(os.path.join(images_path, arquivo),
                    os.path.join(destino_train, arquivo))
    elif "valid" in arquivo:
        shutil.move(os.path.join(images_path, arquivo),
                    os.path.join(destino_valid, arquivo))
    
    print(f"{count}/{total}", end = '\r')

In [31]:
print(len(os.listdir(destino_train)), len(os.listdir(destino_valid)))

97364 19240


In [32]:
train_captions

,ID,Caption
0,ImageCLEFmedical_Caption_2026_train_0,Head CT demonstrating left parotiditis.
1,ImageCLEFmedical_Caption_2026_train_1,Chest X-ray showing enlarged cardiac silhouett...
2,ImageCLEFmedical_Caption_2026_train_2,CT chest axial view showing a huge ascending a...
3,ImageCLEFmedical_Caption_2026_train_3,Acquired renal cysts in end-stage renal failur...
4,ImageCLEFmedical_Caption_2026_train_4,Computed tomography (CT) shows floating thromb...
...,...,...
97359,ImageCLEFmedical_Caption_2026_train_97359,Optical coherence tomography - right eye. The ...
97360,ImageCLEFmedical_Caption_2026_train_97360,Dewarped OCT image of the anterior segment sho...
97361,ImageCLEFmedical_Caption_2026_train_97361,In the first three months after the initial di...
97362,ImageCLEFmedical_Caption_2026_train_97362,B-scan image obtained using a spectral domain ...


In [33]:
from PIL import Image
from torch.utils.data import Dataset
from tqdm import tqdm
import time

def create_dataset_dict(dataset_path, captions_df, image_folder="train"):
    """
    Cria dataset sem carregar imagens na memória (lazy loading).
    As imagens serão carregadas apenas quando acessadas via __getitem__.
    
    Args:
        dataset_path: caminho para o dataset
        captions_df: DataFrame com colunas 'ID' e 'Caption'
        concepts_df: DataFrame com colunas 'ID' e 'CUIs'
        image_folder: nome da pasta com as imagens (ex: 'train', 'valid')
    
    Returns:
        Lista de dicionários com chaves: 'image', 'image_id', 'caption', 'cui'
    """
    dataset_list = []
    
    for idx, row in tqdm(captions_df.iterrows(), total=len(captions_df)):
        image_id = row['ID']
        caption = row['Caption']
        image_path = os.path.join(dataset_path, image_folder, f"{image_id}.jpg")
        
        # Apenas verifica se existe, não carrega
        if not os.path.exists(image_path):
            continue
        
        
        # Salva apenas o caminho, não a imagem
        dataset_list.append({
            'image_path': image_path,  # Caminho em vez da imagem
            'image_id': image_id,
            'caption': caption
        })
    
    return dataset_list

In [35]:
train_images = glob.glob(os.path.join(dataset_path, "train", "*.jpg"))
print(f"Total training images: {len(train_images)}")

test_images = glob.glob(os.path.join(dataset_path, "valid", "*.jpg"))
print(f"Total valid images: {len(test_images)}")

Total training images: 97364
Total valid images: 19240


In [36]:
ds_train = create_dataset_dict(
    dataset_path=dataset_path,
    captions_df=train_captions,
    image_folder="train"
)

ds_valid = create_dataset_dict(
    dataset_path=dataset_path,
    captions_df=valid_captions,
    image_folder="valid"
)

100%|██████████| 19240/19240 [00:00<00:00, 100746.20it/s]


In [40]:
import json

# Criar o diretório se não existir
os.makedirs(f"{BASE_DIR}/artifacts/datasets", exist_ok=True)

# Salvar o dataset em formato JSON
output_path = f"{BASE_DIR}/artifacts/datasets/imageclef2026_train_dataset.json"
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(ds_train, f, ensure_ascii=False, indent=2)

print(f"Train dataset saved to {output_path}")
print(f"Total samples: {len(ds_train)}")

# Salvar o dataset em formato JSON
output_path_valid = f"{BASE_DIR}/artifacts/datasets/imageclef2026_valid_dataset.json"
with open(output_path_valid, 'w', encoding='utf-8') as f:
    json.dump(ds_valid, f, ensure_ascii=False, indent=2)

print(f"valid dataset saved to {output_path_valid}")
print(f"Total samples: {len(ds_valid)}")

Train dataset saved to /home/ia368/projetos/imageclef2026-rag/artifacts/datasets/imageclef2026_train_dataset.json
Total samples: 97364
valid dataset saved to /home/ia368/projetos/imageclef2026-rag/artifacts/datasets/imageclef2026_valid_dataset.json
Total samples: 19240


In [2]:
"""
IC2024 dataset module to run MedSigLip Embeddings
"""
import os
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import argparse
import sys
# Get absolute path to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, project_root)
from utils.f_utils import load_config, load_models, load_dataset
from PIL import Image


# --------------
# Classe Dataset
# --------------

class ImageCLEF24_Dataset(Dataset):
    def __init__(self, split, processor, max_length=64):
        """
        split: split do dataset (ex: ds["train"])
        processor: processor do MedSigLip
        max_length: tamanho máximo do texto
        """
        self.dataset = split
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]
        
        # Carrega a imagem apenas agora
        image = Image.open(sample["image_path"]).convert('RGB')
        text = sample["caption"]

        # Usa o processor do modelo
        # texto (caption) + imagem -> embedding multimodal do MedSigLip ?
        encoding = self.processor(
            text=text,
            images=image,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        # Remove batch dimension criada pelo return_tensors="pt"
        encoding = {k: v.squeeze(0) for k, v in encoding.items()}

        return encoding


# --------------
# Calcula Embeddings
# --------------

In [12]:

config = load_config("../configs/imageclef2026_emb_configs.yaml")
processor, model, device = load_models(config, device=config['device'])

print("Loading Dataset ")
ds = load_dataset(config)

dataset = ImageCLEF24_Dataset(
    split=ds,
    processor=processor,
    max_length=64
)

dataloader = DataLoader(
    dataset,
    batch_size=config['parameters']['batch_size'],
    shuffle=False,
    pin_memory=True
)

for batch in dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}

    image_embeds = model.get_image_features(
        pixel_values=batch["pixel_values"]
    )

    text_embeds = model.get_text_features(
        input_ids=batch["input_ids"]
    )

Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

Loading Dataset 


OutOfMemoryError: CUDA out of memory. Tried to allocate 144.00 MiB. GPU 0 has a total capacity of 31.36 GiB of which 91.31 MiB is free. Process 840885 has 11.80 GiB memory in use. Process 1989741 has 496.00 MiB memory in use. Including non-PyTorch memory, this process has 18.55 GiB memory in use. Of the allocated memory 17.41 GiB is allocated by PyTorch, and 554.66 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [5]:
dataset[0]

{'pixel_values': tensor([[[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.]],
 
         [[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.]],
 
         [[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.]]]),
 'input_ids': tensor([  858,   262,   399,   278, 14136,   698,  4819, 23080,   303,   586,
           447,     1,     1,     1,     1,     1,     1,     1,     1,     1,
   